# 07. 최종 프로젝트 브리핑
> Day 1 · 8H · 소요 약 50분

## 학습 목표

- 최종 프로젝트의 4단계 마일스톤과 평가 루브릭을 이해한다.
- 본인 도메인을 선정하고 제안서 템플릿을 채울 준비를 한다.
- (선택) 제안 스키마 자동 검증기를 실행할 수 있다.

> 이 노트북은 대부분 **마크다운** 으로 구성된 브리핑 자료입니다. 하단에 선택 실행할 수 있는 스키마 검증 코드가 포함되어 있습니다.

## 프로젝트 개요

4일간 배운 기술을 통합해 **자연어 질문 → SQL 생성 → 실행 → 검증 → 답변** 을 자동으로 수행하는 **AI SQL 분석 에이전트** 를 완성합니다.

단순 Text-to-SQL(1회 쿼리)이 아니라, **오류를 스스로 감지하고 재시도하는 에이전틱(Agentic) 워크플로** 가 목표입니다.

## 기술 스택 (고정)

| 영역 | 도구 |
|---|---|
| Database | PostgreSQL (Neon 무료 티어) |
| LLM | OpenAI API (gpt-4o-mini 권장) |
| Text-to-SQL | Vanna.ai (자가학습 루프) |
| RAG 프레임워크 | LlamaIndex · LangChain/LCEL |
| Vector DB | ChromaDB |
| 에이전트 | LangGraph (상태 기반 FSM) |
| 모니터링 | LangSmith (트레이싱) |
| 평가 | Ragas (정량 메트릭) |
| UI (선택) | Gradio |
| 실행 환경 | Google Colab |

## 4단계 마일스톤

| 단계 | 제출 시점 | 제출물 | 형식 |
|---|---|---|---|
| **과제 #1** | Day 2 시작 (9H) | 프로젝트 제안서 | Markdown 문서 |
| **과제 #2** | Day 3 시작 (13H) | 스키마 + 시드 데이터 | Neon PostgreSQL 배포 + DSN 공유 |
| **과제 #3** | Day 4 시작 (21H) | 에이전트 v1 | Colab 노트북 + LangSmith trace URL (10개 질문) |
| **최종 발표** | Day 4 마지막 (24H) | 라이브 데모 + 발표 | 슬라이드 3장 + Ragas 리포트 |

## 과제 #1 — 프로젝트 제안서 양식

아래 구조를 따라 Markdown 문서로 작성하세요.

### 1. 도메인 및 활용 사례

- 선택한 도메인 (1–2문장)
- 대상 사용자와 분석 목적

**도메인 예시**: 전자상거래 / 인사·급여 / IoT 센서 / 학사·수강 / 피트니스 / 게임 로그

### 2. 데이터베이스 스키마

- **테이블 3~5개** (너무 적으면 질문 다양성 부족, 너무 많으면 관리 부담)
- 각 테이블의 컬럼 정의 (타입·PK·FK 명시)
- **`COMMENT ON COLUMN` 필수** — 모든 컬럼에 자연어 설명을 달아야 LLM 이 스키마를 정확히 이해합니다

**스키마 예시:**

```sql
CREATE TABLE patients (
    patient_id   SERIAL PRIMARY KEY,
    name         VARCHAR(100) NOT NULL,
    birth_date   DATE NOT NULL,
    gender       CHAR(1) NOT NULL CHECK (gender IN ('M','F')),
    phone        VARCHAR(20),
    created_at   TIMESTAMP DEFAULT now()
);
COMMENT ON COLUMN patients.patient_id IS '환자 고유 식별자 (자동 증가)';
COMMENT ON COLUMN patients.gender    IS '성별: M=남성, F=여성';
COMMENT ON COLUMN patients.phone     IS '연락처 (선택 입력)';
```

**ERD** — Mermaid 또는 dbdiagram.io 로 간단히 작성.

### 3. 샘플 질문 (정확히 10개)

| 난이도 | 개수 | SQL 요소 |
|---|---|---|
| Easy | 3–4 | `SELECT`, `WHERE`, `ORDER BY` |
| Medium | 3–4 | `GROUP BY`, `JOIN`, `CTE` |
| Hard | 2–3 | 윈도우 함수, 다단계 추론, 서브쿼리 중첩 |

작성 형식 예시:

```
[Easy] Q1: "2024년에 등록된 환자는 몇 명인가요?"
SQL:
  SELECT COUNT(*) FROM patients
  WHERE EXTRACT(YEAR FROM created_at) = 2024;
기대 결과: 단일 숫자 (예: 127)
```

### 4. 데이터 샘플

주요 테이블별 **5–10행** 의 예시 데이터. 시드 데이터의 축약 버전이면 됩니다.

## AI 가독성 스키마 설계 원칙

1. **명확한 네이밍** — `p_id → patient_id`, `dt → visit_date`, `amt → amount`
2. **모든 컬럼에 COMMENT** — 특히 코드값(status, type, severity)에는 반드시 의미 설명
3. **FK 를 명시적으로 선언** — LLM 이 JOIN 경로를 추론하는 유일한 단서
4. **ENUM 보다 룩업 테이블** — `visit_status` 같은 테이블로 값 목록을 조회 가능하게
5. **적정 비정규화** — JOIN 3 단계 이상이면 리포팅 뷰(`vw_*`) 제공

## 질문 10개 난이도 분포 가이드

- **Easy (3–4개)**: 단일 테이블 조회, 단순 필터. 답이 1~수 행.
- **Medium (3–4개)**: 2–3 테이블 JOIN, 집계, CTE.
- **Hard (2–3개)**: 윈도우 함수, 다단계 추론, 서브쿼리 중첩.

**피해야 할 패턴**:
- 10개 모두 Easy → 평가·튜닝 신호가 부족해 Ragas 점수가 의미 없어짐
- 10개 모두 Hard → 에이전트가 전부 실패해 디버깅 포인트를 찾기 어려움
- 동일 형태의 반복("~은 몇 건?" × 10) → 다양성 부족

## 평가 루브릭

| 항목 | 비중 | 핵심 |
|---|---|---|
| 기능·동작 | 30% | 10개 질문 중 **7개 이상** 정답 또는 부분 정답 |
| Ragas 정량 평가 | 25% | Faithfulness, Answer Relevancy, Context Precision/Recall |
| LangGraph 설계 | 20% | 상태 설계, 루프/분기, 에러 처리 |
| 최종 발표 | 15% | 발표 명확성, 데모 품질, 회고의 깊이 |
| 과제 성실도 | 10% | 3건의 과제 기한 내 제출 |

### 등급 기준

| 등급 | 조건 |
|---|---|
| Pass | 제안서 제출 + Neon DB 구축 + 에이전트 실행 + 발표 수행 |
| Strong (75–85%) | 7/10 정답, Faithfulness > 0.7, Answer Relevancy > 0.6 |
| Excellent (90%+) | 9/10 정답, Ragas 전 메트릭 > 0.8, 통찰력 있는 회고, 매끄러운 데모 |

## 최종 발표 가이드 (5–7분)

**슬라이드 3장:**

1. **문제 정의** — 도메인, 분석 목적, 대상 사용자, 주요 질문 유형
2. **아키텍처** — LangGraph 상태 다이어그램, 데이터 흐름(스키마 → 임베딩 → 검색 → SQL 생성 → 실행 → 답변), 도구 선택 이유
3. **결과 및 회고** — Ragas 메트릭(개선 전 vs 후), 성공 2–3건, 실패 1–2건 + 디버깅 과정, 배운 점

**라이브 데모(필수):** 5–7개 질문 실시간 실행, LangSmith trace 1–2건 시연, 성공/실패 사례 각 1건 설명.

**Q&A:** 2분

## 자주 묻는 질문 (FAQ)

- **Q. 테이블을 몇 개까지 만들어야 하나요?** → 3–5개 권장. 3개 미만이면 질문 다양성 부족, 6개 이상이면 관리 부담.
- **Q. 시드 데이터는 실제 데이터여야 하나요?** → 아닙니다. Faker 등으로 생성한 가상 데이터도 OK. 단 질문의 답이 나올 만큼 현실적이어야 합니다.
- **Q. gpt-4를 써도 되나요?** → 가능하지만 비용이 높으므로 gpt-4o-mini 로 개발·테스트 후 최종 평가 시에만 권장.
- **Q. Gradio UI 는 필수인가요?** → 선택. Colab 에서 직접 실행해도 됩니다. 단 Gradio 를 붙이면 데모 인상이 좋아집니다.
- **Q. 팀 프로젝트도 가능한가요?** → 개인 프로젝트입니다. 각자의 도메인과 스키마로 진행합니다.

## (선택) 본인 제안 스키마 자동 검증기

본인의 Neon 인스턴스에 제안 스키마를 올리고 DSN 을 `NEON_DSN` 환경변수에 넣은 뒤, 아래 두 셀을 실행하면 **AI 가독성 점수(0–100)** 와 개선이 필요한 항목 리스트를 출력합니다.

DSN 이 아직 없다면 이 셀들은 건너뛰고 제안서 작성에 집중해도 무방합니다.

In [ ]:
%pip install -q sqlalchemy psycopg2-binary pandas

In [ ]:
# Import + bootstrap only when the student has a Neon DSN.
import os
def _load_optional(key):
    if os.environ.get(key):
        return
    try:
        from google.colab import userdata  # type: ignore
        v = userdata.get(key)
        if v:
            os.environ[key] = v
    except Exception:
        pass

_load_optional("NEON_DSN")
print("NEON_DSN set:", "yes" if os.environ.get("NEON_DSN") else "no (optional)")

In [ ]:
# 스키마 자동 검증기 — 본인 Neon DB 의 스키마가 "AI 가 읽기 좋은가" 를 점수로 채점합니다.
# 채점 기준: PK 존재 여부 + 모든 컬럼에 COMMENT 가 있는지 + 짧은(축약) 이름 사용 여부.
from sqlalchemy import create_engine, inspect, text

def validate_schema(engine, table_names):
    """Score a proposed schema for AI-friendliness."""
    inspector = inspect(engine)
    # report 는 결과를 누적할 dict — 함수 끝에서 통째로 반환.
    report = {"tables": {}, "score": 0, "issues": []}
    total_points = 0    # 만점 카운트 — 컬럼 수 + 테이블당 5점(PK 보너스)
    earned_points = 0   # 실제 획득 점수

    for table in table_names:
        # 테이블별 진단 항목을 모아 둘 mini-report.
        table_report = {"columns": 0, "comments": 0, "fks": 0, "pk": False}

        # ---- 컬럼 수 ----
        columns = inspector.get_columns(table)
        table_report["columns"] = len(columns)
        total_points += len(columns)   # 컬럼당 1점이 만점에 누적

        # ---- PK 존재 여부 ----
        pk = inspector.get_pk_constraint(table)
        if pk and pk["constrained_columns"]:
            table_report["pk"] = True
            earned_points += 5
        else:
            # 발견된 문제는 issues 리스트에 누적해 마지막에 사람이 보기 좋게 출력.
            report["issues"].append(f"[warn] {table}: PRIMARY KEY 없음")
        total_points += 5

        # ---- FK 카운트 (점수 가산은 안 하고 표시용으로만) ----
        fks = inspector.get_foreign_keys(table)
        table_report["fks"] = len(fks)

        # ---- COMMENT 조회 ----
        # information_schema.columns + col_description 함수로 한 번에 컬럼명 + 코멘트를 받는다.
        # f-string 으로 테이블명을 직접 끼워 넣고 있으므로 외부 입력엔 절대 사용 금지(여기는 학습용).
        with engine.connect() as conn:
            comments = conn.execute(text(f"""
                SELECT column_name,
                       col_description('{table}'::regclass, ordinal_position) AS comment
                FROM information_schema.columns
                WHERE table_name = '{table}'
                ORDER BY ordinal_position
            """)).fetchall()

        # 각 컬럼마다 코멘트 유무를 점수화 — 있으면 +1.
        for col_name, comment in comments:
            if comment:
                table_report["comments"] += 1
                earned_points += 1
            else:
                report["issues"].append(f"[warn] {table}.{col_name}: COMMENT 없음")

        # ---- 짧은(축약) 컬럼명 경고 ----
        # 'id' 는 관용적으로 OK 처리. 그 외 3 글자 이하 이름은 의미 추론이 어려워 경고.
        short_names = [
            col["name"] for col in columns
            if len(col["name"]) <= 3 and col["name"] not in ("id",)
        ]
        if short_names:
            report["issues"].append(
                f"[warn] {table}: 짧은 컬럼명 {short_names} — 명시적 이름 권장"
            )

        report["tables"][table] = table_report

    # 0으로 나누는 ZeroDivisionError 를 막기 위해 max(total, 1) 사용.
    report["score"] = round(earned_points / max(total_points, 1) * 100, 1)
    return report

In [ ]:
# Example usage — adjust table_names to your own schema.
# if os.environ.get("NEON_DSN"):
#     engine = create_engine(os.environ["NEON_DSN"])
#     report = validate_schema(engine, ["patients", "doctors", "visits", "diagnoses", "departments"])
#     print(f"Score: {report['score']} / 100\n")
#     for table, info in report["tables"].items():
#         pk_icon = "ok" if info["pk"] else "no"
#         ratio   = f"{info['comments']}/{info['columns']}"
#         print(f"  {table}: PK={pk_icon} FK={info['fks']} COMMENT={ratio}")
#     if report["issues"]:
#         print(f"\nIssues ({len(report['issues'])}):")
#         for issue in report["issues"][:15]:
#             print(f"  {issue}")
# else:
#     print("(Set NEON_DSN to run the validator on your own schema.)")
print("Validator function defined. See the commented-out example above.")

## 실습 과제

다음 1 가지 실습을 직접 작성해 보세요. (정답 코드는 의도적으로 비워 두었습니다.)

### 1. 본인 도메인의 ERD 초안 작성 (Mermaid)

**본인 도메인의 ERD 초안을 Mermaid로 작성하세요.**

1. 도메인을 정하세요 (5분)
2. 핵심 테이블 3~5개의 이름과 주요 컬럼을 구상하세요 (10분)
3. 아래 템플릿을 복사하여 채워보세요:

```
erDiagram
    테이블A ||--o{ 테이블B : "관계"
    테이블A {
        serial id PK
        varchar name "이름"
    }
    테이블B {
        serial id PK
        int a_id FK
        varchar description "설명"
    }
```

4. 질문 10개를 난이도별로 작성하세요 (20분)
5. 나머지는 집에서 완성하여 **내일(Day 2) 9H에 제출**


In [ ]:
# ============================================================
# 실습 과제 — 본인 도메인 ERD 초안 작성
# ============================================================

# 실습 1: 본인 도메인의 ERD 초안을 Mermaid로 작성
# TODO: 도메인을 정하고 테이블 3~5개와 주요 컬럼을 떠올린 뒤,
#       아래 템플릿을 복사해 채우세요. 결과는 https://mermaid.live 에
#       붙여넣어 시각화로 확인할 수 있습니다. (코드 실행 불필요)
#
# erDiagram
#     테이블A ||--o{ 테이블B : "관계"
#     테이블A {
#         serial id PK
#         varchar name "이름"
#     }
#     테이블B {
#         serial id PK
#         int a_id FK
#         varchar description "설명"
#     }
#
# 여기에 작성하세요. (Markdown 셀로 옮겨도 좋습니다)


## 제출 방법 & 다음 노트북

**과제 #1 (제안서)** — 오늘 밤 완성하여 내일 (Day 2 · 9H) 시작 시 제출합니다. `PROJECT_BRIEF.md` 의 템플릿을 복사해 본인 내용으로 채우면 됩니다.

### 다음 노트북에서는…

Day 2 의 첫 노트북 **`08_text_to_sql_advanced.ipynb`** 에서 `NLSQLTableQueryEngine` 의 내부 프롬프트·Few-shot·`ObjectIndex` 로 정확도를 끌어올립니다. 그 전에 9H 에는 여러분의 **제안서를 3인 1조로 피어리뷰** 하니, 제안서를 반드시 준비해 오세요.